In [1]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling,BitsAndBytesConfig
from datasets import load_dataset
import torch
import evaluate
from trl import SFTConfig, SFTTrainer
from peft import get_peft_model, LoraConfig, TaskType

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
dataset = load_dataset("lavita/medical-qa-datasets", "all-processed")['train']
dataset

Dataset({
    features: ['instruction', 'input', 'output', '__index_level_0__'],
    num_rows: 239357
})

In [4]:
dataset = dataset.filter(lambda x: x['output'] is not None and x['output'].strip() != "")

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    "gpt2-large",
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("gpt2-large")

In [6]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

In [7]:
def format_prompt(instruction, input_text, output_text=None, tokenizer=None):
    if input_text and input_text.strip():
        prompt = f"### Instruction: {instruction}\n### Input: {input_text}\n### Response:"
    else:
        prompt = f"### Instruction: {instruction}\n### Response:"
    if output_text is not None:
        if tokenizer is not None:
            prompt += f" {output_text}{tokenizer.eos_token}"
        else:
            prompt += f" {output_text}"
    return prompt

def tokenize_supervised(example, tokenizer, max_length=512):
    full_prompt = format_prompt(example["instruction"], example["input"], example["output"], tokenizer=tokenizer)
    prompt_only = format_prompt(example["instruction"], example["input"])

    prompt_only_ids = tokenizer(prompt_only, truncation=True, max_length=max_length)["input_ids"]
    prompt_len = len(prompt_only_ids)

    tokenized = tokenizer(
        full_prompt,
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    labels = tokenized["input_ids"].copy()
    if prompt_len >= max_length:
        return None
    labels[:prompt_len] = [-100] * prompt_len

    if all(l == -100 for l in labels):
        return None
    tokenized["labels"] = labels
    return tokenized

In [8]:
response_lengths = [len(response.split()) for response in dataset['output']]

max_length_response = max(response_lengths)

max_length_response

7731

In [9]:
def tokenize_fn(example):
        return tokenize_supervised(example, tokenizer)
tokenized_dataset = dataset.map(tokenize_fn, remove_columns=dataset.column_names)

In [10]:
dataset_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset_split['train'].select(range(100000))
test_dataset = dataset_split['test'].select(range(20000))

In [11]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [12]:
model = model.to(device)
print("Model device:", next(model.parameters()).device)

prompt = dataset[500]['instruction'] + dataset[500]['input']
inputs = tokenizer(prompt, return_tensors="pt", padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}
for k, v in inputs.items():
    print(f"{k}: {v.device}")

with torch.no_grad():
    gen_tokens = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )
gen_text = tokenizer.decode(gen_tokens[0], skip_special_tokens=True)
response = gen_text.split("### Response:")[-1].strip()
print(">>> Prompt:\n", prompt)
print(">>> Base model output:\n", response)
print(">>> Expected output:\n", dataset[500]['output'])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Model device: cuda:0
input_ids: cuda:0
attention_mask: cuda:0
>>> Prompt:
 Answer this question truthfullyWhat is the explanation for Ultraviolet? Could you recommend some additional reading materials about the subject?
>>> Base model output:
 Answer this question truthfullyWhat is the explanation for Ultraviolet? Could you recommend some additional reading materials about the subject?

Answer this question truthfullyWhat is the explanation for Ultraviolet? Could you recommend some additional reading materials about the subject?

Answer this question truthfullyWhat is the explanation for Ultraviolet? Could you recommend some additional reading materials about the subject?

Answer this question truthfullyWhat is the explanation for Ultraviolet? Could you recommend some additional reading materials about the subject?

Answer this question truthfullyWhat is the explanation for Ultraviolet? Could you recommend some additional reading materials about the subject?

Answer this question truth

In [13]:
sacrebleu = evaluate.load("sacrebleu")
results_base = sacrebleu.compute(predictions=[response], references=[dataset[500]['output']])
print("BLEU keys:", list(results_base.keys()))
print("BLEU score:", round(results_base["score"], 1))

BLEU keys: ['score', 'counts', 'totals', 'precisions', 'bp', 'sys_len', 'ref_len']
BLEU score: 0.3


In [14]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.2,  # Increase dropout to avoid overfitting
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

In [15]:
training_args = SFTConfig(
    output_dir="./out",
    num_train_epochs=1,
    save_strategy="epoch",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    max_seq_length=1024,
    logging_steps=100,
    logging_first_step=True,
    do_eval=True,
)

trainer = SFTTrainer(
    model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    args=training_args,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

In [ ]:
trainer.save_model("/tmp")
tokenizer.save_pretrained("/tmp")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("/tmp")
prompt = format_prompt(dataset[0]['instruction'], dataset[0]['input'])
inputs = tokenizer(prompt, return_tensors="pt", padding=True)
with torch.no_grad():
    gen_tokens = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )
gen_text = tokenizer.decode(gen_tokens[0], skip_special_tokens=True)
response = gen_text.split("### Response:")[-1].strip()
print("Fine-tuned model output:\n", response)
